# FlowLens v2: native interpretation and frozen evidence

This notebook inspects the published release and development examples. It does not reopen or rescore the raw final test.


In [ ]:
from pathlib import Path
import json
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/interpretable_detector.py').exists())
evaluation = json.loads((ROOT / 'models/v2/final_evaluation.json').read_text())
development = json.loads((ROOT / 'models/v2/development_report.json').read_text())


In [ ]:
pd.DataFrame({name: values for name, values in development['finalist_validation'].items()}).T


In [ ]:
pd.DataFrame(evaluation['selected_model']['per_family']).T


## Native explanations of actual deployed predictions

A single sigmoid calibration link preserves the additive log-odds structure. Native final terms plus their calibrated intercept reconstruct the final probability.


In [ ]:
import sys, joblib
import numpy as np
from scipy.special import expit
sys.path.insert(0, str(ROOT))
from src.interpretable_detector import native_record_explanation
bundle = joblib.load(ROOT / 'models/v2/detector.joblib')
sample = pd.read_csv(ROOT / 'data/rich_flow_sample.csv')
explanation = native_record_explanation(bundle, sample)
np.testing.assert_allclose(expit(explanation['final_intercept'] + explanation['final_contributions'].sum(axis=1)), explanation['probability'])
pd.DataFrame({'term': explanation['terms'], 'contribution_log_odds': explanation['final_contributions'][0]}).sort_values('contribution_log_odds')


## Reproduction

Use the README commands to recreate prepared inputs and write a separate experiment directory. Published-split replays are reproductions, not fresh blind evaluations. See docs/MODEL_CARD_V2.md for the dataset scope and original evaluation design.
